# Qwen3-4B (wo-reasoning) — evaluate the ViNumQA SFT adapter

Loads the LoRA adapter from `qwen3-4b-stf-wo-reasoning-trace.ipynb` and scores Program
Accuracy / Execution Accuracy on `test.json`, using the same prompt and parser as every other
notebook in this repo.

Separate from the training notebook on purpose. That notebook calls
`FastLanguageModel.get_peft_model()` immediately after loading, which stacks a freshly
initialised LoRA on top of whatever was loaded -- pointing it at a trained adapter and running
its cells would score a half-random model, and it would re-run `trainer.train()` as well.

Before running: attach the dataset holding the training run's output and set `ADAPTER_DIR`.

### Installation

In [ ]:
%%capture
import os, re

# Let the CUDA allocator grow segments rather than fragment into blocks it
# cannot reuse. Generation allocates and frees KV-cache buffers of varying
# size on every step, which is exactly the pattern that fragments the heap --
# the OOM that killed an earlier run reported 384 MiB reserved-but-unusable.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups (incl. Kaggle)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -q tabulate
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Load the fine-tuned adapter

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096   # matches the training run

# Folder holding the adapter, i.e. the one containing adapter_model.safetensors
# -- not `qwen3-4b-vinumqa-sft`, which holds mid-training trainer checkpoints.
ADAPTER_DIR = "/kaggle/input/<your-dataset-slug>/qwen3-4b-vinumqa-sft-adapter"

import os
assert os.path.isdir(ADAPTER_DIR), (
    f"{ADAPTER_DIR} not found -- set ADAPTER_DIR to the folder shown in the Data panel."
)
print("Found adapter:", sorted(os.listdir(ADAPTER_DIR))[:6])

# from_pretrained on an adapter directory pulls the base model named in
# adapter_config.json and applies the LoRA weights on top. No get_peft_model
# call here -- that would add a second, untrained adapter.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print("Adapter loaded.")

### Data + prompt (identical to the training notebook)

In [ ]:
import pandas as pd
from tabulate import tabulate

test_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/test.json')
print(f"test={len(test_df)}")

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "post_text_processed", "input_question",
             "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "post_text", "question", "program", "answer"]
    return df

test_df = process_split(test_df)
test_df["generated_program"] = ""
test_df.head(3)

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(val1, val2, val3, ...) -> sum of the values
8. table_average(val1, val2, val3, ...) -> arithmetic mean of the values
9. table_max(val1, val2, val3, ...) -> maximum of the values
10. table_min(val1, val2, val3, ...) -> minimum of the values

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- Do not use square brackets `[]` inside table-type functions (e.g. write table_max(1, 2, 3), not table_max([1, 2, 3])).
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use \'none\'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

# Same prompt format as the 0-shot/1-shot/sft notebooks, so results are
# comparable across all experiments (only the training regime differs).

### PA / EA

In [ ]:
"""
Parser + evaluation utilities for FinQA/VLSP-2025 style computation programs.
(Same implementation as the 0-shot/1-shot/sft notebooks.)
"""

import re
from typing import List, Tuple, Union, Optional

VALID_OPERATORS = {
    "add", "subtract", "multiply", "divide", "exp", "greater",
    "table_sum", "table_average", "table_max", "table_min",
}
_BINARY_OPS = {"add", "subtract", "multiply", "divide", "exp", "greater"}
_TABLE_OPS = {"table_sum", "table_average", "table_max", "table_min"}
_NUM_RE = re.compile(r"^-?\d+(\.\d+)?$")
_REF_RE = re.compile(r"^#(\d+)$")


def _parse_numeric_literal(raw: str) -> float:
    cleaned = raw.replace(",", "").replace("$", "").strip()
    is_percent = cleaned.endswith("%")
    if is_percent:
        cleaned = cleaned[:-1].strip()
    if not _NUM_RE.match(cleaned):
        raise ValueError(f"Cannot parse numeric argument: '{raw}'")
    value = float(cleaned)
    return value / 100.0 if is_percent else value


def extract_program(raw_text: str) -> str:
    text = raw_text.strip()
    text = re.sub(r"```[a-zA-Z]*", "", text)
    text = text.replace("```", "").strip()
    op_names = "|".join(sorted(VALID_OPERATORS, key=len, reverse=True))
    pattern = rf"\b(?:{op_names})\([^()]*\)"
    matches = re.findall(pattern, text)
    if matches:
        return ", ".join(m.strip() for m in matches)
    return text


def _split_top_level(s: str, sep: str = ",") -> List[str]:
    parts, depth, current = [], 0, []
    for ch in s:
        if ch == "(":
            depth += 1; current.append(ch)
        elif ch == ")":
            depth -= 1; current.append(ch)
        elif ch == sep and depth == 0:
            parts.append("".join(current)); current = []
        else:
            current.append(ch)
    if current:
        parts.append("".join(current))
    return [p.strip() for p in parts if p.strip() != ""]


def parse_program(program_str: str) -> List[Tuple[str, List[str]]]:
    program_str = program_str.strip().rstrip(",").strip()
    if not program_str:
        raise ValueError("Empty program string.")
    steps: List[Tuple[str, List[str]]] = []
    call_pattern = re.compile(r"\s*([a-zA-Z_]+)\(([^()]*)\)\s*")
    pos = 0
    text = program_str
    while pos < len(text):
        m = call_pattern.match(text, pos)
        if not m:
            raise ValueError(f"Malformed program near: '{text[pos:pos+30]}...'")
        op = m.group(1).strip()
        if op not in VALID_OPERATORS:
            raise ValueError(f"Unknown operator: '{op}'")
        args = _split_top_level(m.group(2), sep=",")
        steps.append((op, args))
        pos = m.end()
        if pos < len(text) and text[pos] == ",":
            pos += 1
    if not steps:
        raise ValueError("No valid steps parsed.")
    return steps


def _resolve_arg(arg: str, results: List[float]) -> Optional[float]:
    arg = arg.strip()
    ref_match = _REF_RE.match(arg)
    if ref_match:
        idx = int(ref_match.group(1))
        if idx >= len(results):
            raise ValueError(f"Reference #{idx} used before step {idx} was computed.")
        return results[idx]
    if arg.lower() == "none":
        return None
    return _parse_numeric_literal(arg)


def execute_program(program_str: str) -> Tuple[float, List[float]]:
    steps = parse_program(program_str)
    results: List[float] = []
    for op, raw_args in steps:
        vals = [_resolve_arg(a, results) for a in raw_args]
        if op in _BINARY_OPS:
            if len(vals) != 2:
                raise ValueError(f"Operator '{op}' expects 2 args, got {len(vals)}.")
            a, b = vals
            if a is None or b is None:
                raise ValueError(f"Operator '{op}' received a 'none' operand.")
            if op == "add": r = a + b
            elif op == "subtract": r = a - b
            elif op == "multiply": r = a * b
            elif op == "divide":
                if b == 0:
                    raise ValueError("Division by zero.")
                r = a / b
            elif op == "exp": r = a ** b
            elif op == "greater": r = 1.0 if a > b else 0.0
        elif op in _TABLE_OPS:
            if len(vals) < 1:
                raise ValueError(f"Operator '{op}' requires at least 1 arg.")
            if any(v is None for v in vals):
                raise ValueError(f"Operator '{op}' received a 'none' operand.")
            if op == "table_sum": r = sum(vals)
            elif op == "table_average": r = sum(vals) / len(vals)
            elif op == "table_max": r = max(vals)
            elif op == "table_min": r = min(vals)
        else:
            raise ValueError(f"Unknown operator: '{op}'")
        results.append(r)
    return results[-1], results


def _normalize_program(program_str: str) -> List[Tuple[str, Tuple[str, ...]]]:
    steps = parse_program(program_str)
    normalized = []
    for op, args in steps:
        norm_args = []
        for a in args:
            a = a.strip()
            if _REF_RE.match(a) or a.lower() == "none":
                norm_args.append(a.lower())
            else:
                try:
                    norm_args.append(f"{round(_parse_numeric_literal(a), 6)}")
                except ValueError:
                    norm_args.append(a)
        normalized.append((op, tuple(norm_args)))
    return normalized


def compute_program_accuracy(generated_program, gold_program, alternative_gold_programs=None) -> float:
    try:
        gen_norm = _normalize_program(generated_program)
    except ValueError:
        return 0.0
    gold_candidates = [gold_program] + (alternative_gold_programs or [])
    for gold in gold_candidates:
        try:
            gold_norm = _normalize_program(gold)
        except ValueError:
            continue
        if gen_norm == gold_norm:
            return 1.0
    return 0.0


def compute_execution_accuracy(generated_program, gold_answer, rel_tol: float = 1e-3, abs_tol: float = 1e-4) -> float:
    try:
        predicted, _ = execute_program(generated_program)
    except (ValueError, ZeroDivisionError, OverflowError):
        return 0.0
    try:
        gold = _parse_numeric_literal(str(gold_answer))
    except ValueError:
        return 0.0
    if abs(predicted - gold) <= max(abs_tol, rel_tol * abs(gold)):
        return 1.0
    return 0.0


def evaluate_dataframe(df, generated_col="generated_program", gold_program_col="program",
                        gold_answer_col="answer", extract_first=True):
    df = df.copy()
    pa_scores, ea_scores = [], []
    for _, row in df.iterrows():
        raw_generated = row[generated_col]
        generated = extract_program(raw_generated) if extract_first else raw_generated
        pa = compute_program_accuracy(generated, row[gold_program_col])
        ea = compute_execution_accuracy(generated, row[gold_answer_col])
        pa_scores.append(pa)
        ea_scores.append(ea)
    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores
    summary = {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }
    return df, summary

In [ ]:
import gc
from pathlib import Path
from tqdm import tqdm

QWEN3_THINK_END_TOKEN_ID = 151668  # "</think>"

# One prompt at a time, no padding. Batching several prompts requires left
# padding, and Unsloth's fast inference path derives token positions from the
# cache length rather than from the attention mask, so padded rows decode at the
# wrong positions -- measured on this exact model, that cost 0.6419 -> 0.6338 PA.
# Extra votes come from num_return_sequences, which share one prompt and so need
# no padding, and keep memory bounded by N_VOTES rather than N_VOTES x batch
# (a 4 x 5 = 20-sequence batch OOMed a 14.5 GiB T4).
#
# N_VOTES = 1 decodes greedily, on purpose. Qwen3 ships generation_config with
# do_sample=True, so leaving the arguments off -- as the original loop did --
# makes every pass a fresh random draw: the same adapter scored 0.6338 and
# 0.6217 PA on two runs. A 0.02 gap between configurations then says nothing.
# Greedy is deterministic, so repeated runs and different configurations become
# comparable. It does mean the earlier 0.6419 (sampled) is not a like-for-like
# reference; treat greedy numbers as their own series.
N_VOTES = 1
EVAL_MAX_NEW_TOKENS = 256   # programs are short; unchanged from the original run
TEMPERATURE = 0.7           # only used when N_VOTES > 1
TOP_P = 0.95

# Checkpoints live under /kaggle/working, never in an attached input dataset --
# a stale eval_partial_*.csv from an earlier run must not be resumed.
#
# The filename encodes the decoding settings, because resuming across a change
# of settings silently reports the old numbers: switching k=1 from sampling to
# greedy left a full checkpoint in place, so the loop generated 0 samples and
# re-scored the sampled predictions while claiming to be greedy. A different
# decode now means a different file, so it regenerates.
DECODE_TAG = "greedy" if N_VOTES == 1 else f"sample-t{TEMPERATURE}-p{TOP_P}"
CKPT_PATH = Path(f"/kaggle/working/eval_partial_k{N_VOTES}_{DECODE_TAG}.csv")

FORCE_REGENERATE = False   # set True to ignore any checkpoint and start over

test_df["generated_program"] = ""
if CKPT_PATH.exists() and not FORCE_REGENERATE:
    _done = pd.read_csv(CKPT_PATH, index_col=0).fillna("")
    test_df.loc[_done.index, "generated_program"] = _done["generated_program"].values
    print(f"Resumed {(test_df['generated_program'] != '').sum()} / {len(test_df)} "
          f"from {CKPT_PATH.name}")
else:
    print(f"Starting fresh -> {CKPT_PATH.name}")


def build_prompt(row):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )


def strip_think(output_ids):
    """Qwen3 emits an empty <think></think> pair even with thinking disabled."""
    ids = list(output_ids)
    try:
        cut = len(ids) - ids[::-1].index(QWEN3_THINK_END_TOKEN_ID)
    except ValueError:
        cut = 0
    return tokenizer.decode(ids[cut:], skip_special_tokens=True).strip()


def vote(candidates):
    """Most common program, keyed by normalised form so that cosmetically
    different but structurally identical programs share a vote."""
    cleaned = [extract_program(c) for c in candidates if c and c.strip()]
    if not cleaned:
        return ""
    keyed = {}
    for c in cleaned:
        try:
            key = str(_normalize_program(c))
        except Exception:
            key = c.strip()
        keyed.setdefault(key, []).append(c)
    best = max(keyed, key=lambda k: (len(keyed[k]), -cleaned.index(keyed[k][0])))
    return keyed[best][0]


n_unparsable = 0
todo = [i for i in test_df.index if not str(test_df.at[i, "generated_program"]).strip()]

for n, df_index in enumerate(tqdm(todo, desc=f"Generating (k={N_VOTES})")):
    enc = tokenizer([build_prompt(test_df.loc[df_index])], return_tensors="pt").to(model.device)

    gen_kwargs = dict(max_new_tokens=EVAL_MAX_NEW_TOKENS)
    if N_VOTES > 1:
        gen_kwargs.update(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
                          num_return_sequences=N_VOTES)
    else:
        gen_kwargs.update(do_sample=False)

    try:
        with torch.no_grad():
            out = model.generate(**enc, **gen_kwargs)
        prompt_len = enc["input_ids"].shape[1]
        gen_only = [r[prompt_len:].tolist() for r in out]
        cands = [strip_think(g) for g in gen_only]
        # With enable_thinking=False the template puts the empty <think></think>
        # pair in the *prompt*, so the model generates the program directly and
        # no closing tag appears in its output -- counting those would flag all
        # 497 samples. What actually matters is whether the output parses into a
        # program at all; anything else is a truncated or malformed generation.
        chosen = vote(cands) if N_VOTES > 1 else cands[0]
        test_df.at[df_index, "generated_program"] = chosen
        try:
            parse_program(extract_program(chosen))
        except Exception:
            n_unparsable += 1
        del enc, out
    except torch.cuda.OutOfMemoryError:
        print(f"  OOM on index {df_index}; leaving it blank (scored 0).")
        test_df.at[df_index, "generated_program"] = ""

    if n % 25 == 0:
        test_df[["generated_program"]].to_csv(CKPT_PATH)
        gc.collect()
        torch.cuda.empty_cache()

test_df[["generated_program"]].to_csv(CKPT_PATH)
print(f"Done. {(test_df['generated_program'] != '').sum()} / {len(test_df)} generated "
      f"({len(todo)} newly generated this run).")
if not todo:
    print("!! Nothing was generated -- every prediction came from the checkpoint, so "
          "the scores below describe the earlier run, not the current settings. "
          "Set FORCE_REGENERATE = True to redo them.")
print(f"Outputs that did not parse into a program: {n_unparsable} / {len(todo)} "
      f"-- these score 0 on both metrics; if the count is high, the generations "
      f"are being cut off and EVAL_MAX_NEW_TOKENS needs raising.")

In [ ]:
df_scored, summary = evaluate_dataframe(test_df)
print(summary)

# The 0.6419 / 0.6439 reference was produced with sampling left on, so it is one
# draw rather than a fixed value -- the same adapter has since scored 0.6338 and
# 0.6217 across runs. Greedy decoding here is deterministic, so these numbers can
# be compared against each other and against future greedy runs, but a direct
# subtraction from the sampled reference would be reading noise as signal.
print()
print(f"{'setting':<44}{'PA':>10}{'EA':>10}")
print("-" * 64)
print(f"{'earlier run (2048, sampled -- one draw)':<44}{0.6419:>10.4f}{0.6439:>10.4f}")
print(f"{'this adapter (4096, greedy -- deterministic)':<44}"
      f"{summary['program_accuracy']:>10.4f}{summary['execution_accuracy']:>10.4f}")
print()
print("Greedy is reproducible: re-running this cell gives the same number, so any "
      "later change can be attributed to that change rather than to sampling luck.")

In [ ]:
# ---------------------------------------------------------------- diagnostic
# 41 outputs failed to parse. Before reading that as model error, check what
# they actually are: the repo's scorer matches operator calls with `[^()]*`,
# which cannot span a nested parenthesis, so a *correct* answer referencing a
# table row whose label contains brackets -- table_min(ROE (%), none),
# table_max(EPS (VND), none) -- is rejected before it is ever compared to gold.
# Feeding the gold programs themselves through this scorer caps it at PA 0.930 /
# EA 0.875 for exactly this reason.
#
# This cell only measures how much of the gap is scorer artefact. It does not
# change the reported metric: every number in the repo uses the strict parser,
# and switching would break comparability with the published rows.
import re as _re

def _paren_aware_parse(prog):
    """Same grammar, but match brackets by depth instead of by regex."""
    text = prog.strip().rstrip(",").strip()
    steps, pos = [], 0
    name = _re.compile(r"\s*([a-zA-Z_]+)\(")
    while pos < len(text):
        m = name.match(text, pos)
        if not m:
            raise ValueError("malformed")
        op = m.group(1)
        if op not in VALID_OPERATORS:
            raise ValueError("unknown op")
        start, depth, i = m.end() - 1, 0, m.end() - 1
        while i < len(text):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    break
            i += 1
        else:
            raise ValueError("unbalanced")
        steps.append((op, _split_top_level(text[start + 1:i], ",")))
        pos = i + 1
        while pos < len(text) and text[pos] in ", ":
            pos += 1
    if not steps:
        raise ValueError("empty")
    return steps

def _norm(prog):
    out = []
    for op, args in _paren_aware_parse(prog):
        na = []
        for a in args:
            a = a.strip()
            if _REF_RE.match(a) or a.lower() == "none":
                na.append(a.lower())
            else:
                try:
                    na.append(f"{round(_parse_numeric_literal(a), 6)}")
                except ValueError:
                    na.append(a)
        out.append((op, tuple(na)))
    return out

strict_fail, recovered, examples = 0, 0, []
for _, row in test_df.iterrows():
    gen = extract_program(row["generated_program"])
    try:
        parse_program(gen)
        continue                      # strict parser coped; nothing to check
    except Exception:
        strict_fail += 1
    raw = str(row["generated_program"]).strip()
    try:
        ok = _norm(raw) == _norm(row["program"])
    except Exception:
        ok = False
    if ok:
        recovered += 1
        if len(examples) < 3:
            examples.append((row["question"][:70], row["program"], raw))

print(f"outputs the strict parser rejected      : {strict_fail}")
print(f"  ...of which actually match gold exactly: {recovered}")
print(f"  ...genuinely wrong or malformed        : {strict_fail - recovered}")
print()
print(f"reported PA                 : {summary['program_accuracy']:.4f}")
print(f"PA if those were credited   : "
      f"{(summary['program_accuracy'] * len(test_df) + recovered) / len(test_df):.4f}")
print(f"scorer ceiling (gold vs gold): 0.9296")
print()
for q, gold, got in examples:
    print(f"  Q: {q}")
    print(f"     gold  : {gold}")
    print(f"     model : {got}")
